# Multiple Linear Regression: Prediksi Jarak Berdasarkan Waktu Tempuh

Notebook ini memakai dataset `DATA PERJALANAN.csv` untuk mempelajari multiple linear regression. Model memakai beberapa fitur perjalanan, yaitu `WAKTU`, `KECEPATAN`, `JUMLAH_LAMPU_MERAH`, `JUMLAH_BELOKAN`, dan `KONDISI_JALAN`, untuk memprediksi `JARAK`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression


## 1. Membaca Data CSV

Dataset `DATA PERJALANAN.csv` berisi data perjalanan dengan kolom:

- `JARAK`: jarak tempuh dalam kilometer
- `WAKTU`: waktu tempuh dalam menit
- `KECEPATAN`: kecepatan rata-rata dalam km/jam
- `JUMLAH_LAMPU_MERAH`: jumlah lampu merah yang dilalui
- `JUMLAH_BELOKAN`: jumlah belokan selama perjalanan
- `KONDISI_JALAN`: kondisi jalan (0=buruk, 1=sedang, 2=baik)


In [ ]:
csv_path = Path("../dataset/DATA PERJALANAN.csv")

if not csv_path.exists():
    csv_path = Path("dataset/DATA PERJALANAN.csv")

if not csv_path.exists():
    # Buat dataset sintetis jika file tidak ada
    import numpy as np
    import os

    np.random.seed(42)
    n = 100
    waktu = np.random.randint(5, 120, n)               # menit
    kecepatan = np.random.randint(20, 80, n)           # km/jam
    lampu_merah = np.random.randint(0, 15, n)          # jumlah
    belokan = np.random.randint(0, 20, n)              # jumlah
    kondisi_jalan = np.random.randint(0, 3, n)         # 0/1/2
    jarak = (
        0.8 * (waktu / 60) * kecepatan
        - 0.3 * lampu_merah
        - 0.1 * belokan
        + 0.5 * kondisi_jalan
        + np.random.normal(0, 1.5, n)
    ).clip(0.5)

    os.makedirs("dataset", exist_ok=True)
    df = pd.DataFrame({
        "JARAK": jarak.round(2),
        "WAKTU": waktu,
        "KECEPATAN": kecepatan,
        "JUMLAH_LAMPU_MERAH": lampu_merah,
        "JUMLAH_BELOKAN": belokan,
        "KONDISI_JALAN": kondisi_jalan,
    })
    df.to_csv("dataset/DATA PERJALANAN.csv", index=False)
    csv_path = Path("dataset/DATA PERJALANAN.csv")
    print("Dataset sintetis berhasil dibuat.")

df = pd.read_csv(csv_path)
df.head()


## 2. Memisahkan Fitur dan Target

`X` adalah fitur yang dipakai untuk prediksi. `y` adalah nilai yang ingin diprediksi.


In [ ]:
fitur = ["WAKTU", "KECEPATAN", "JUMLAH_LAMPU_MERAH", "JUMLAH_BELOKAN", "KONDISI_JALAN"]
X = df[fitur]
y = df["JARAK"]

print("Fitur X:")
display(X.head())

print("Target y (jarak dalam km):")
display(y.head())


## 3. Melatih Model Linear Regression


In [ ]:
model = LinearRegression()
model.fit(X, y)

print(f"Intercept: {model.intercept_:.4f} km")
for nama_fitur, koefisien in zip(fitur, model.coef_):
    print(f"Koefisien {nama_fitur}: {koefisien:.4f}")


Rumus linear regression yang dipelajari model:

`jarak_km = intercept + koef_WAKTU*WAKTU + koef_KECEPATAN*KECEPATAN + koef_LAMPU_MERAH*JUMLAH_LAMPU_MERAH + koef_BELOKAN*JUMLAH_BELOKAN + koef_KONDISI_JALAN*KONDISI_JALAN`

Hasil prediksi model dibaca dalam satuan **kilometer**.


## 4. Memprediksi Jarak Perjalanan Baru


In [ ]:
perjalanan_baru = pd.DataFrame(
    [[45, 60, 5, 8, 1]],
    columns=fitur,
)
prediksi_jarak = model.predict(perjalanan_baru)[0]

print("Data perjalanan baru:")
display(perjalanan_baru)
print(f"Prediksi jarak: {prediksi_jarak:.2f} km")


## 5. Visualisasi dengan Matplotlib

Karena model memakai banyak fitur, visualisasi dibuat dengan membandingkan jarak asli dan jarak prediksi. Jika titik dekat dengan garis merah, prediksi model semakin dekat dengan jarak asli.


In [ ]:
jarak_prediksi = model.predict(X)

plt.scatter(y, jarak_prediksi, color="blue", alpha=0.6, label="Data perjalanan")
plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    color="red",
    label="Prediksi ideal",
)

plt.title("Jarak Asli vs Jarak Prediksi")
plt.xlabel("Jarak asli (km)")
plt.ylabel("Jarak prediksi (km)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
